# Morphology-aware Old Tupi tokenizer/canonicalizer proof of concept

This notebook is a repeatable research experiment for a tokenizer that learns from the pydicate + oldtupicorpus ecosystem.

The target shape remains:

```text
raw Old Tupi surface text
-> normalization / orthography handling
-> morpheme and allomorph segmentation
-> grammar-aware canonical stream
-> reversible-ish inspection
```

This is not ordinary BPE. BPE would learn frequent string fragments, but it would not know that `r` can be a pluriform prefix, that a surface token has person/role features, or that pydicate-rendered analyses can teach canonical morphology. Here the pydicate corpus is the teacher, and the experiment should improve when the corpus grows.

## 1. Setup

Run from the `oldtupicorpus` repo root. The setup cell finds the root, adds `../nhe-enga/tupi` and `../nhe-enga/pydicate` to `sys.path`, then checks the local imports used by corpus generation.

In [1]:
from __future__ import annotations

import json
import random
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "tokenizer").is_dir() and (candidate / "historic").is_dir():
            return candidate
    raise RuntimeError("Could not find oldtupicorpus repo root from the current directory")


ROOT = find_repo_root()
NHE_ENGA = (ROOT.parent / "nhe-enga").resolve()
PATHS_TO_ADD = [ROOT, NHE_ENGA / "tupi", NHE_ENGA / "pydicate"]

for path in reversed(PATHS_TO_ADD):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f"Repo root: {ROOT}")
for path in PATHS_TO_ADD:
    print(f"sys.path entry {'OK' if path.exists() else 'MISSING'}: {path}")

from tokenizer.morph_poc_utils import (
    FactorizationConfig,
    LexiconAwareMorphBaseline,
    MorphBaseline,
    append_jsonl,
    generate_navarro_lexicon_rows,
    build_morph_rows,
    count_by,
    evaluate_prediction_fn,
    format_metrics_table,
    inspect_tokens as inspect_tokens_with_registry,
    iter_jsonl,
    load_json,
    load_jsonl,
    load_navarro_lexicon,
    load_registry,
    mismatch_summary,
    normalize_surface,
    token_prf,
    utc_now_iso,
    write_morph_dataset,
)

IMPORT_STATUS = {}

try:
    import tupi
    from tupi import TupiAntigo
    IMPORT_STATUS["tupi"] = True
    print(f"OK: import tupi -> {getattr(tupi, '__file__', '(namespace package)')}")
except Exception as exc:
    IMPORT_STATUS["tupi"] = False
    print(f"FAIL: import tupi: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang"] = True
    print("OK: from pydicate.lang.tupilang import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang"] = False
    print(f"FAIL: from pydicate.lang.tupilang import *: {type(exc).__name__}: {exc}")

try:
    from pydicate.lang.tupilang.pos import *  # noqa: F401,F403
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = True
    print("OK: from pydicate.lang.tupilang.pos import *")
except Exception as exc:
    IMPORT_STATUS["pydicate.lang.tupilang.pos"] = False
    print(f"FAIL: from pydicate.lang.tupilang.pos import *: {type(exc).__name__}: {exc}")

if not all(IMPORT_STATUS.values()):
    print("\nSome imports failed. Existing artifacts can still be inspected, but rebuilding may fail until ../nhe-enga imports work.")

Repo root: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/oldtupicorpus
sys.path entry OK: /Users/kian/code/nhe-enga/tupi
sys.path entry OK: /Users/kian/code/nhe-enga/pydicate
OK: import tupi -> /Users/kian/code/nhe-enga/tupi/tupi/__init__.py
OK: from pydicate.lang.tupilang import *
OK: from pydicate.lang.tupilang.pos import *


## 2. Experiment configuration

These are the main knobs. Set `FORCE_REBUILD = True` when you have added pydicate-encoded data to `historic/` or `synthetic/` and want the corpus, factorized dataset, model, metrics, and history to reflect it.

In [2]:
# Corpus rebuild controls
FORCE_REBUILD = True
INCLUDE_SYNTHETIC = True
LABEL_FROM_ANNOTATED = True
ORTH_EXPAND = ["POTIGUARA", "TUPINAMBA", "SEM_DIACRITICO"]
ORTH_EXPAND_ALL = False
ORTH_WORKERS = 1
ORTH_BATCH_SIZE = 200
BUILD_LOG_EVERY = 10000

# Navarro lexical-prior controls
USE_NAVARRO_LEXICON = True
ADD_LEXEME_TOKENS = True
NAVARRO_CLASSES = ["noun", "verb", "postposition", "adverb", "pronoun"]
GENERATE_NAVARRO_LEXICON_ROWS = True
GENERATE_NAVARRO_POSTPOSITION_COMBOS = True
MAX_NAVARRO_AUGMENT_ROWS = 10000
NAVARRO_ROOT_BONUS = 6.0
NAVARRO_POSTPOSITION_BONUS = 5.0
NAVARRO_FEATURE_BONUS = 1.5
RAW_PENALTY = 8.0
SEGMENT_PENALTY = 0.2

# Factorized target controls
DROP_FEATURE_PREFIXES = {"DEEPEST_NODE"}
DROP_FEATURES = {"DIRECT"}
KEEP_ROOT_FEATURE = True
USE_EXPLICIT_S_IDS = False

# Dataset and split controls
MAX_EXAMPLES = 5000
RANDOM_SEED = 42
MIN_INPUT_LEN = 1
MAX_SRC_LEN = 220
MAX_TGT_LEN = 320
DEV_FRACTION = 0.2
EVAL_LIMIT = 200
QUALITATIVE_EXAMPLES = 10

# Neural model controls
TRAIN_MODEL = True
SAVE_CHECKPOINT = True
CHECKPOINT_PATH = "tokenizer/output/morph_tokenizer_poc.pt"
EPOCHS = 3
BATCH_SIZE = 16
EMBED_DIM = 64
HIDDEN_DIM = 128
LEARNING_RATE = 3e-3
TEACHER_FORCING = 0.85
MAX_DECODE_LEN = 160

# Experiment logging
WRITE_EXPERIMENT_HISTORY = True
HISTORY_PATH = "tokenizer/output/morph_experiment_history.jsonl"

BUILD_CONFIG = {
    "force_rebuild": FORCE_REBUILD,
    "include_synthetic": INCLUDE_SYNTHETIC,
    "label_from_annotated": LABEL_FROM_ANNOTATED,
    "orth_expand": ORTH_EXPAND,
    "orth_expand_all": ORTH_EXPAND_ALL,
    "orth_workers": ORTH_WORKERS,
    "orth_batch_size": ORTH_BATCH_SIZE,
    "build_log_every": BUILD_LOG_EVERY,
}

LEXICON_CONFIG = {
    "use_navarro_lexicon": USE_NAVARRO_LEXICON,
    "add_lexeme_tokens": ADD_LEXEME_TOKENS,
    "navarro_classes": NAVARRO_CLASSES,
    "generate_navarro_lexicon_rows": GENERATE_NAVARRO_LEXICON_ROWS,
    "generate_navarro_postposition_combos": GENERATE_NAVARRO_POSTPOSITION_COMBOS,
    "max_navarro_augment_rows": MAX_NAVARRO_AUGMENT_ROWS,
    "navarro_root_bonus": NAVARRO_ROOT_BONUS,
    "navarro_postposition_bonus": NAVARRO_POSTPOSITION_BONUS,
    "navarro_feature_bonus": NAVARRO_FEATURE_BONUS,
    "raw_penalty": RAW_PENALTY,
    "segment_penalty": SEGMENT_PENALTY,
}

TRAINING_CONFIG = {
    "max_examples": MAX_EXAMPLES,
    "random_seed": RANDOM_SEED,
    "min_input_len": MIN_INPUT_LEN,
    "max_src_len": MAX_SRC_LEN,
    "max_tgt_len": MAX_TGT_LEN,
    "dev_fraction": DEV_FRACTION,
    "train_model": TRAIN_MODEL,
    "save_checkpoint": SAVE_CHECKPOINT,
    "checkpoint_path": CHECKPOINT_PATH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "learning_rate": LEARNING_RATE,
    "teacher_forcing": TEACHER_FORCING,
}

FACTORIZATION_CONFIG = FactorizationConfig(
    drop_feature_prefixes=set(DROP_FEATURE_PREFIXES),
    drop_features=set(DROP_FEATURES),
    keep_root_feature=KEEP_ROOT_FEATURE,
    use_explicit_s_ids=USE_EXPLICIT_S_IDS,
)

pprint({"build": BUILD_CONFIG, "lexicon": LEXICON_CONFIG, "factorization": FACTORIZATION_CONFIG.to_json(), "training": TRAINING_CONFIG})

{'build': {'build_log_every': 10000,
           'force_rebuild': True,
           'include_synthetic': True,
           'label_from_annotated': True,
           'orth_batch_size': 200,
           'orth_expand': ['POTIGUARA', 'TUPINAMBA', 'SEM_DIACRITICO'],
           'orth_expand_all': False,
           'orth_workers': 1},
 'factorization': {'drop_feature_prefixes': ['DEEPEST_NODE'],
                   'drop_features': ['DIRECT'],
                   'keep_root_feature': True,
                   'use_explicit_s_ids': False},
 'lexicon': {'add_lexeme_tokens': True,
             'generate_navarro_lexicon_rows': True,
             'generate_navarro_postposition_combos': True,
             'max_navarro_augment_rows': 10000,
             'navarro_classes': ['noun',
                                 'verb',
                                 'postposition',
                                 'adverb',
                                 'pronoun'],
             'navarro_feature_bonus': 1.5,
         

## 3. Rebuild or load tokenizer artifacts

The notebook still relies on the existing scripts for source discovery and canonical ID generation:

- `tokenizer/build_corpus_json.py`
- `tokenizer/rawgrammarpair.py`

The difference from the first proof of concept is that rebuild behavior is explicit and controlled by the config above.

In [3]:
OUT_DIR = ROOT / "tokenizer" / "output"
CORPUS_JSONL = OUT_DIR / "corpus.jsonl"
CANONICAL_IO = OUT_DIR / "canonical_io.jsonl"
TOKENS_JSON = OUT_DIR / "annotated_tokens.json"
TAGS_JSON = OUT_DIR / "annotated_tags.json"
SUBTAGS_JSON = OUT_DIR / "annotated_subtags.json"
TOKEN_PAIRS_JSON = OUT_DIR / "annotated_token_pairs.json"
VARIANTS_JSON = OUT_DIR / "annotated_token_variants.json"
MORPH_IO = OUT_DIR / "morph_io.jsonl"
MORPH_VOCAB = OUT_DIR / "morph_vocab.json"
MORPH_META = OUT_DIR / "morph_dataset_meta.json"
HISTORY_FILE = ROOT / HISTORY_PATH
CHECKPOINT_FILE = ROOT / CHECKPOINT_PATH


def run_repo_command(args: list[str]) -> None:
    print("$", " ".join(args))
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-6000:])
    if result.stderr:
        print(result.stderr[-6000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")


OUT_DIR.mkdir(parents=True, exist_ok=True)

build_cmd = [
    "python3",
    "tokenizer/build_corpus_json.py",
    "--out_jsonl",
    "tokenizer/output/corpus.jsonl",
]
if INCLUDE_SYNTHETIC:
    build_cmd.append("--include-synthetic")
if LABEL_FROM_ANNOTATED:
    build_cmd.append("--label-from-annotated")
if ORTH_EXPAND:
    build_cmd.extend(["--orth-expand", *ORTH_EXPAND])
if ORTH_EXPAND_ALL:
    build_cmd.append("--orth-expand-all")
if ORTH_WORKERS:
    build_cmd.extend(["--orth-workers", str(ORTH_WORKERS)])
if ORTH_BATCH_SIZE:
    build_cmd.extend(["--orth-batch-size", str(ORTH_BATCH_SIZE)])
if BUILD_LOG_EVERY:
    build_cmd.extend(["--log-every", str(BUILD_LOG_EVERY)])

rawgrammar_cmd = [
    "python3",
    "tokenizer/rawgrammarpair.py",
    "--in_json",
    "tokenizer/output/corpus.jsonl",
    "--out_dir",
    "tokenizer/output",
]
if BUILD_LOG_EVERY:
    rawgrammar_cmd.extend(["--log-every", str(BUILD_LOG_EVERY)])

if FORCE_REBUILD or not CORPUS_JSONL.exists():
    run_repo_command(build_cmd)
else:
    print(f"Loading existing {CORPUS_JSONL.relative_to(ROOT)}")

core_outputs = [CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON]
if FORCE_REBUILD or not all(path.exists() for path in core_outputs):
    run_repo_command(rawgrammar_cmd)
else:
    print("Loading existing canonical IO and registries")

for path in [CORPUS_JSONL, CANONICAL_IO, TOKENS_JSON, TAGS_JSON, SUBTAGS_JSON, TOKEN_PAIRS_JSON, VARIANTS_JSON]:
    print(f"{path.relative_to(ROOT)}: {'present' if path.exists() else 'missing'}")

$ python3 tokenizer/build_corpus_json.py --out_jsonl tokenizer/output/corpus.jsonl --include-synthetic --label-from-annotated --orth-expand POTIGUARA TUPINAMBA SEM_DIACRITICO --orth-workers 1 --orth-batch-size 200 --log-every 10000
[corpus] rows=20000 skipped=0 rate=6267.7/s
[corpus] rows=60000 skipped=0 rate=6656.1/s
[corpus] rows=140000 skipped=0 rate=7058.7/s
[corpus] rows=210000 skipped=0 rate=7147.6/s
[corpus] rows=260000 skipped=0 rate=7217.5/s
[corpus] rows=270000 skipped=0 rate=7194.6/s
[corpus] rows=280000 skipped=0 rate=7201.4/s
[corpus] rows=360000 skipped=0 rate=7099.5/s
[corpus] rows=390000 skipped=0 rate=7125.9/s
[corpus] rows=410000 skipped=0 rate=7107.8/s
[corpus] rows=420000 skipped=0 rate=7111.4/s
[corpus] rows=440000 skipped=0 rate=7095.3/s
[corpus] rows=500000 skipped=0 rate=7262.6/s
[corpus] rows=520000 skipped=0 rate=7257.0/s
[corpus] rows=600000 skipped=0 rate=7262.5/s
[corpus] rows=650000 skipped=0 rate=7254.9/s
[corpus] rows=670000 skipped=0 rate=7226.9/s
[corp

## 4. Factorize and write the reusable morph dataset

This section converts opaque canonical IDs into the model-facing stream and writes reusable artifacts:

- `tokenizer/output/morph_io.jsonl`
- `tokenizer/output/morph_vocab.json`
- `tokenizer/output/morph_dataset_meta.json`

`ROOT` is kept by default as `<G:ROOT>` because it is linguistically useful. Generated structure such as `DEEPEST_NODE_*` and debug relation `DIRECT` are dropped by default.

In [4]:
corpus_rows = load_jsonl(CORPUS_JSONL)
canonical_rows = load_jsonl(CANONICAL_IO)
token_items, id_to_morpheme, morpheme_to_id = load_registry(TOKENS_JSON, "value")
tag_items, id_to_tag, tag_to_id = load_registry(TAGS_JSON, "tag")
subtag_items, id_to_subtag, subtag_to_id = load_registry(SUBTAGS_JSON, "subtag")
token_pairs = load_json(TOKEN_PAIRS_JSON, default=[])
token_variants = load_json(VARIANTS_JSON, default=[])

base_morph_rows, morph_build_stats = build_morph_rows(
    corpus_rows,
    canonical_rows,
    id_to_tag,
    id_to_subtag,
    FACTORIZATION_CONFIG,
)

navarro_entries = []
navarro_aug_rows = []
if USE_NAVARRO_LEXICON:
    navarro_entries = load_navarro_lexicon(NHE_ENGA, NAVARRO_CLASSES)
    if GENERATE_NAVARRO_LEXICON_ROWS:
        navarro_aug_rows = generate_navarro_lexicon_rows(
            navarro_entries,
            generate_postposition_combos=GENERATE_NAVARRO_POSTPOSITION_COMBOS,
            max_rows=MAX_NAVARRO_AUGMENT_ROWS,
        )

morph_rows = base_morph_rows + navarro_aug_rows
morph_vocab, morph_meta = write_morph_dataset(
    MORPH_IO,
    MORPH_VOCAB,
    MORPH_META,
    morph_rows,
    corpus_rows,
    {"corpus_build": BUILD_CONFIG, "lexicon": LEXICON_CONFIG},
    FACTORIZATION_CONFIG,
    lexicon_entries=navarro_entries,
    add_lexeme_tokens=ADD_LEXEME_TOKENS,
)

BASELINE = MorphBaseline(
    id_to_morpheme=id_to_morpheme,
    id_to_tag=id_to_tag,
    canonical_rows=canonical_rows,
    token_variants=token_variants,
    factorization_config=FACTORIZATION_CONFIG,
)

LEXICON_BASELINE = None
if USE_NAVARRO_LEXICON:
    LEXICON_BASELINE = LexiconAwareMorphBaseline(
        id_to_morpheme=id_to_morpheme,
        id_to_tag=id_to_tag,
        canonical_rows=canonical_rows,
        token_variants=token_variants,
        factorization_config=FACTORIZATION_CONFIG,
        lexicon_entries=navarro_entries,
        navarro_root_bonus=NAVARRO_ROOT_BONUS,
        navarro_postposition_bonus=NAVARRO_POSTPOSITION_BONUS,
        navarro_feature_bonus=NAVARRO_FEATURE_BONUS,
        raw_penalty=RAW_PENALTY,
        segment_penalty=SEGMENT_PENALTY,
    )


def baseline_tokenize(text: str) -> list[str]:
    return BASELINE.tokenize(text)


def baseline_tokenize_batch(texts: list[str]) -> list[list[str]]:
    return BASELINE.tokenize_batch(texts)


def baseline_raw_rate(pred_tokens: list[str]) -> float:
    return BASELINE.raw_rate(pred_tokens)


def lexicon_baseline_tokenize(text: str) -> list[str]:
    if LEXICON_BASELINE is None:
        return baseline_tokenize(text)
    return LEXICON_BASELINE.tokenize(text)


def lexicon_baseline_tokenize_with_trace(text: str) -> tuple[list[str], list[dict]]:
    if LEXICON_BASELINE is None:
        return baseline_tokenize(text), []
    return LEXICON_BASELINE.tokenize_with_trace(text)


def inspect_tokens(tokens: list[str]) -> list[dict]:
    lexicon_by_token = LEXICON_BASELINE.lexicon_by_token if LEXICON_BASELINE is not None else {}
    return inspect_tokens_with_registry(tokens, id_to_morpheme, lexicon_by_token)


sample_for_raw = base_morph_rows[: min(1000, len(base_morph_rows))]
baseline_raw_oov_rate = (
    sum(baseline_raw_rate(baseline_tokenize(row["input"])) for row in sample_for_raw) / max(1, len(sample_for_raw))
)
lexicon_raw_oov_rate = (
    sum(baseline_raw_rate(lexicon_baseline_tokenize(row["input"])) for row in sample_for_raw) / max(1, len(sample_for_raw))
)
corpus_summary = morph_meta["counts"]

print("Corpus and dataset counts")
print(f"  corpus rows:              {len(corpus_rows)}")
print(f"  canonical rows:           {len(canonical_rows)}")
print(f"  base morph rows:          {len(base_morph_rows)}")
print(f"  Navarro augment rows:     {len(navarro_aug_rows)}")
print(f"  total morph rows:         {len(morph_rows)}")
print(f"  historic rows:            {corpus_summary['historic_rows']}")
print(f"  synthetic rows:           {corpus_summary['synthetic_rows']}")
print(f"  orthographic variant rows:{corpus_summary['orthographic_variant_rows']}")
print(f"  Navarro lexicon entries:  {len(navarro_entries)}")
print(f"  M-token count:            {len(morph_vocab['m_tokens'])}")
print(f"  LEX-token count:          {len(morph_vocab.get('lex_tokens', []))}")
print(f"  G-feature count:          {len(morph_vocab['g_tokens'])}")
print(f"  baseline RAW/OOV rate:    {baseline_raw_oov_rate:.3%} on {len(sample_for_raw)} corpus rows")
print(f"  lexicon RAW/OOV rate:     {lexicon_raw_oov_rate:.3%} on {len(sample_for_raw)} corpus rows")
print(f"  stale variant rows ignored by baseline: {BASELINE.stale_variant_rows}")
print("\nCounts by corpus:")
pprint(corpus_summary["by_corpus"])
print("\nMorph rows by corpus:")
pprint(morph_meta.get("morph_counts", {}).get("by_corpus", {}))
print("\nCounts by orthography:")
pprint(corpus_summary["by_orth"])
print("\nWrote:")
for path in [MORPH_IO, MORPH_VOCAB, MORPH_META]:
    print(f"  {path.relative_to(ROOT)}")

if navarro_entries:
    print("\nNavarro samples for ka'a and pe:")
    for entry in navarro_entries:
        if entry.normalized_surface in {"ka'a", "pe"}:
            pprint(entry)

Corpus and dataset counts
  corpus rows:              2014384
  canonical rows:           2014384
  base morph rows:          2014384
  Navarro augment rows:     10000
  total morph rows:         2024384
  historic rows:            358
  synthetic rows:           2014026
  orthographic variant rows:1319996
  Navarro lexicon entries:  6882
  M-token count:            5200
  LEX-token count:          6882
  G-feature count:          104
  baseline RAW/OOV rate:    2.391% on 1000 corpus rows
  lexicon RAW/OOV rate:     3.326% on 1000 corpus rows
  stale variant rows ignored by baseline: 0

Counts by corpus:
{'historic': 358, 'synthetic': 2014026}

Morph rows by corpus:
{'historic': 358, 'lexicon': 10000, 'synthetic': 2014026}

Counts by orthography:
{'NAVARRO': 694388,
 'POTIGUARA': 539119,
 'SEM_DIACRITICO': 642854,
 'TUPINAMBA': 138023}

Wrote:
  tokenizer/output/morph_io.jsonl
  tokenizer/output/morph_vocab.json
  tokenizer/output/morph_dataset_meta.json

Navarro samples for ka'a and p

In [5]:
def compact_tokens(tokens: list[str] | str, max_tokens: int = 80) -> str:
    if isinstance(tokens, str):
        tokens = tokens.split()
    if len(tokens) <= max_tokens:
        return " ".join(tokens)
    return " ".join(tokens[:max_tokens]) + f" ... (+{len(tokens) - max_tokens} tokens)"


print("corpus.jsonl examples")
for row in corpus_rows[:3]:
    pprint(row)

print("\nmorph_io.jsonl examples")
for row in morph_rows[:3]:
    pprint(row)

print("\nregistry examples")
print("M:")
pprint(token_items[:5])
print("T:")
pprint(tag_items[:5])
print("S:")
pprint(subtag_items[:8])
if token_variants:
    print("variants:")
    pprint(token_variants[:5])

corpus.jsonl examples
{'anotated': 'Santa '
             "Cruz[DEEPEST_NODE_7:DIRECT:OBJECT:PROPER_NOUN]r[DEEPEST_NODE_5:PLURIFORM_PREFIX:R]a'ang[DEEPEST_NODE_6:ROOT]ab[DEEPEST_NODE_5:FACILITY_SUFFIX]a[CONSONANT_ENDING:DEEPEST_NODE_5:NOUN:POSSESSOR:SUBSTANTIVE_SUFFIX] "
             'r[DEEPEST_NODE_4:PLURIFORM_PREFIX:R]esé[DEEPEST_NODE_4:POSTPOSITION] '
             'oré[1ppe:DEEPEST_NODE_3:OBJECT:PRONOUN]pysyrõ[DEEPEST_NODE_1:ROOT] '
             'îepé[2ps:DEEPEST_NODE_1:OBJECT_1P:PRONOUN:SUBJECT] '
             'Tupã[DEEPEST_NODE_9:PROPER_NOUN] '
             'oré[1ppe:DEEPEST_NODE_11:OBJECT:POSSESSIVE_PRONOUN:PRONOUN] '
             'îar[DEEPEST_NODE_10:ROOT:VOCATIVE] '
             "oré[1ppe:DEEPEST_NODE_15:OBJECT:PRONOUN]amotar[DEEPEST_NODE_14:ROOT]e'ym[DEEPEST_NODE_13:NEGATION_SUFFIX]bar[ABSOLUTE_AGENT_SUFFIX:DEEPEST_NODE_13]a[CONSONANT_ENDING:DEEPEST_NODE_13:SUBSTANTIVE_SUFFIX] "
             'suí[DEEPEST_NODE_12:POSTPOSITION]',
 'corpus': 'historic',
 'index': 0,
 'label': "San

## 5. Navarro lexical prior

Observed M-token matching is useful, but it is still fragile: it can prefer locally valid chunks that are not the intended lexical root. The Navarro dictionary adds an underlying lexeme layer. Dictionary roots and postpositions are privileged during segmentation, while corpus-derived M tokens remain available.

The target vocabulary can now contain three complementary token families:

- `<M:...>` for observed surface/morpheme realization
- `<LEX:NAVARRO:...>` for underlying dictionary lexemes
- `<G:...>` for grammar features

The lexicon-aware baseline uses global dynamic programming instead of greedy longest-match, so a whole-word analysis like `ka'a + pe` can beat a sequence of smaller local chunks.

In [6]:
LEXICON_TEST_STRINGS = [
    "ka'ape",
    "ka'a pe",
    "ka'ápe",
]

for text in LEXICON_TEST_STRINGS:
    old_pred = baseline_tokenize(text)
    lex_pred, trace = lexicon_baseline_tokenize_with_trace(text)
    print("\n" + "=" * 80)
    print("INPUT:", text)
    print("OLD BASELINE:     ", compact_tokens(old_pred, max_tokens=80))
    print("LEXICON BASELINE: ", compact_tokens(lex_pred, max_tokens=80))
    print("inspection:")
    pprint(inspect_tokens(lex_pred)[:20])
    print("score trace:")
    pprint(trace)


INPUT: ka'ape
OLD BASELINE:      <M:001842> <RAW:'> <M:000746>
LEXICON BASELINE:  <M:001842> <LEX:NAVARRO:noun:'a> <G:NOUN> <G:ROOT> <G:LOCATIVE> <M:000027> <G:2pp> <G:SUBJECT_PREFIX>
inspection:
[{'grammar': [], 'raw': False, 'surface': 'ká', 'token': '<M:001842>'},
 {'classname': 'noun',
  'definition': "(s.) (em compos. somente): cabeça: Aî'a-kok. - Apoio sua "
                "cabeça. (VLB, I, 18); Aî'a-su'u - Mordo-lhe a cabeça. (VLB, "
                'I, 94) ● \'a-pixapaba - ferida da cabeça "Às vezes se usa '
                'deste nome nas feridas que se dão por outras partes, fora da '
                'ca...',
  'grammar': ['NOUN', 'ROOT', 'LOCATIVE'],
  'raw': False,
  'source': 'navarro',
  'surface': "'a",
  'token': "<LEX:NAVARRO:noun:'a>"},
 {'grammar': ['2pp', 'SUBJECT_PREFIX'],
  'raw': False,
  'surface': 'pe',
  'token': '<M:000027>'}]
score trace:
[{'cumulative_score': 7.86117135969092,
  'emitted_tokens': ['<M:001842>'],
  'end': 2,
  'global_end': 2,
  'global_st

## 6. Baseline tokenizer/canonicalizer

The plain baseline remains as a control model. It is intentionally transparent:

- Unicode NFC + whitespace normalization
- longest-match segmentation within each whitespace word
- longer known morphemes are preferred
- `annotated_token_variants.json` contributes variant-to-canonical candidates when the IDs exist in the current registry
- unknown chunks are emitted as `<RAW:...>`
- grammar features are estimated from the most frequent observed feature sequence for the M-token

The lexicon-aware baseline keeps this corpus-derived layer and adds Navarro candidates plus global DP scoring.

In [7]:
for text in [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "xe rera",
    "xerera",
]:
    base_pred = baseline_tokenize(text)
    lex_pred = lexicon_baseline_tokenize(text)
    print("\n---")
    print("INPUT:", text)
    print("BASELINE:        ", compact_tokens(base_pred, max_tokens=100))
    print("LEXICON BASELINE:", compact_tokens(lex_pred, max_tokens=100))
    print(f"baseline RAW/OOV: {baseline_raw_rate(base_pred):.2%}")
    print(f"lexicon RAW/OOV:  {baseline_raw_rate(lex_pred):.2%}")
    print("lexicon inspection:")
    pprint(inspect_tokens(lex_pred)[:20])


---
INPUT: amém
BASELINE:         <M:000024> <G:AMEN> <G:INTERJECTION>
LEXICON BASELINE: <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000094> <G:ABSOLUTE> <G:M> <G:PLURIFORM_PREFIX> <LEX:NAVARRO:noun:é> <G:NOUN> <G:ROOT> <M:000094> <G:ABSOLUTE> <G:M> <G:PLURIFORM_PREFIX>
baseline RAW/OOV: 0.00%
lexicon RAW/OOV:  0.00%
lexicon inspection:
[{'grammar': ['1ps', 'SUBJECT_PREFIX'],
  'raw': False,
  'surface': 'a',
  'token': '<M:000006>'},
 {'grammar': ['ABSOLUTE', 'M', 'PLURIFORM_PREFIX'],
  'raw': False,
  'surface': 'm',
  'token': '<M:000094>'},
 {'classname': 'noun',
  'definition': '(s.) - coisa distinta, coisa própria, coisa diferente; (adj.) '
                '- próprio, vário, outro, diferente, não comum aos outros, '
                'particular; separado (e não de parceria com alguém): Tub-é. - '
                'Tem outro pai (isto é, diferente do pai de seu irmão); ...',
  'grammar': ['NOUN', 'ROOT'],
  'raw': False,
  'source': 'navarro',
  'surface': 'é',
  'token': '<LEX:NAVARR

## 7. Train/dev split from `morph_io.jsonl`

The training source is now the reusable factorized dataset, not notebook-local variables. The split is deterministic with `RANDOM_SEED`; rows that are empty or too long are skipped and counted.

In [8]:
PAD = "<PAD>"
BOS = "<BOS>"
EOS = "<EOS>"
UNK = "<UNK>"

all_morph_rows = load_jsonl(MORPH_IO)
loaded_vocab = load_json(MORPH_VOCAB)

skip_counts = Counter()
eligible_rows = []
for row in all_morph_rows:
    input_text = normalize_surface(str(row.get("input", "")))
    target_tokens = str(row.get("target", "")).split()
    if len(input_text) < MIN_INPUT_LEN:
        skip_counts["too_short_input"] += 1
        continue
    if len(input_text) > MAX_SRC_LEN:
        skip_counts["too_long_input"] += 1
        continue
    if not target_tokens:
        skip_counts["empty_target"] += 1
        continue
    if len(target_tokens) + 2 > MAX_TGT_LEN:
        skip_counts["too_long_target"] += 1
        continue
    next_row = dict(row)
    next_row["input"] = input_text
    next_row["target_tokens"] = target_tokens
    eligible_rows.append(next_row)

rng = random.Random(RANDOM_SEED)
rng.shuffle(eligible_rows)
selected_rows = eligible_rows[:MAX_EXAMPLES]
split_at = max(1, int(len(selected_rows) * (1.0 - DEV_FRACTION))) if len(selected_rows) > 1 else len(selected_rows)
train_examples = selected_rows[:split_at]
dev_examples = selected_rows[split_at:] or selected_rows[: min(10, len(selected_rows))]

src_chars = sorted({ch for row in train_examples for ch in row["input"]})
src_itos = [PAD, BOS, EOS, UNK] + src_chars
src_stoi = {tok: i for i, tok in enumerate(src_itos)}

tgt_itos = loaded_vocab["tokens"]
for required in [PAD, BOS, EOS, UNK]:
    if required not in tgt_itos:
        tgt_itos.insert(0, required)
tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}

SRC_PAD_IDX = src_stoi[PAD]
TGT_PAD_IDX = tgt_stoi[PAD]


def encode_source(text: str) -> list[int]:
    text = normalize_surface(text)[:MAX_SRC_LEN]
    ids = [src_stoi[BOS]]
    ids.extend(src_stoi.get(ch, src_stoi[UNK]) for ch in text)
    ids.append(src_stoi[EOS])
    return ids


def encode_target(tokens: list[str]) -> list[int]:
    clipped = tokens[: MAX_TGT_LEN - 2]
    ids = [tgt_stoi[BOS]]
    ids.extend(tgt_stoi.get(tok, tgt_stoi[UNK]) for tok in clipped)
    ids.append(tgt_stoi[EOS])
    return ids


def decode_target(ids: list[int]) -> list[str]:
    out = []
    for idx in ids:
        tok = tgt_itos[int(idx)]
        if tok == EOS:
            break
        if tok not in {PAD, BOS}:
            out.append(tok)
    return out


print(f"morph rows loaded: {len(all_morph_rows)}")
print(f"eligible rows:     {len(eligible_rows)}")
print(f"selected rows:     {len(selected_rows)} / MAX_EXAMPLES={MAX_EXAMPLES}")
print(f"train/dev:         {len(train_examples)} / {len(dev_examples)}")
print(f"skipped rows:      {dict(skip_counts)}")
print(f"source char vocab: {len(src_itos)}")
print(f"target vocab:      {len(tgt_itos)}")
if selected_rows:
    print("\nexample input:", selected_rows[0]["input"])
    print("example target:", compact_tokens(selected_rows[0]["target_tokens"], max_tokens=100))

morph rows loaded: 2024384
eligible rows:     2024384
selected rows:     5000 / MAX_EXAMPLES=5000
train/dev:         4000 / 1000
skipped rows:      {}
source char vocab: 44
target vocab:      12191

example input: pee tapeie'akok ume
example target: <M:000356> <G:2pp> <G:PRONOUN> <G:SUBJECT> <M:000357> <G:CONSONANT> <G:PERMISSIVE_PREFIX> <M:000027> <G:2pp> <G:SUBJECT_PREFIX> <M:000236> <G:OBJECT> <G:OBJECT_PREFIX> <G:REFLEXIVE> <G:refl> <M:000368> <M:000251> <G:NEGATION_PARTICLE> <G:UME>


## 8. Small PyTorch encoder-decoder with attention

This remains intentionally small and local. It downloads no pretrained model.

Install dependencies if needed:

```bash
python3 -m pip install torch numpy notebook ipykernel
```

If `TRAIN_MODEL = False` and `CHECKPOINT_PATH` exists, the cell loads the checkpoint for inference.

In [9]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f"torch: {torch.__version__}")
except ModuleNotFoundError:
    TORCH_AVAILABLE = False
    print("Torch is not installed. Install it to run neural training:")
    print("  python3 -m pip install torch numpy notebook ipykernel")

NEURAL_READY = False
neural_model = None
checkpoint_saved_path = None

if TORCH_AVAILABLE and selected_rows:
    def choose_device():
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")

    device = choose_device()
    print("device:", device)

    def pad_sequences(seqs: list[list[int]], pad_idx: int):
        max_len = max(len(seq) for seq in seqs)
        tensor = torch.full((len(seqs), max_len), pad_idx, dtype=torch.long)
        for i, seq in enumerate(seqs):
            tensor[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        return tensor

    def make_batches(examples: list[dict], batch_size: int, shuffle: bool = True):
        rows = examples[:]
        if shuffle:
            random.shuffle(rows)
        for start in range(0, len(rows), batch_size):
            batch = rows[start : start + batch_size]
            src = pad_sequences([encode_source(row["input"]) for row in batch], SRC_PAD_IDX)
            tgt = pad_sequences([encode_target(row["target_tokens"]) for row in batch], TGT_PAD_IDX)
            yield src, tgt, batch

    class Encoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)

        def forward(self, src):
            emb = self.embedding(src)
            outputs, hidden = self.gru(emb)
            hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)
            hidden0 = torch.tanh(self.hidden_proj(hidden_cat)).unsqueeze(0)
            return outputs, hidden0

    class AttentionDecoder(nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, enc_dim: int, pad_idx: int):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
            self.enc_proj = nn.Linear(enc_dim, hidden_dim)
            self.gru = nn.GRU(embed_dim + enc_dim, hidden_dim, batch_first=True)
            self.out = nn.Linear(hidden_dim + enc_dim + embed_dim, vocab_size)

        def step(self, prev_tok, hidden, enc_outputs, src_mask):
            emb = self.embedding(prev_tok).unsqueeze(1)
            projected = self.enc_proj(enc_outputs)
            scores = torch.bmm(projected, hidden[-1].unsqueeze(2)).squeeze(2)
            scores = scores.masked_fill(~src_mask, -1e9)
            weights = torch.softmax(scores, dim=1).unsqueeze(1)
            context = torch.bmm(weights, enc_outputs)
            output, hidden = self.gru(torch.cat([emb, context], dim=2), hidden)
            logits = self.out(torch.cat([output.squeeze(1), context.squeeze(1), emb.squeeze(1)], dim=1))
            return logits, hidden

    class Seq2Seq(nn.Module):
        def __init__(self, src_vocab: int, tgt_vocab: int, embed_dim: int, hidden_dim: int):
            super().__init__()
            self.encoder = Encoder(src_vocab, embed_dim, hidden_dim, SRC_PAD_IDX)
            self.decoder = AttentionDecoder(tgt_vocab, embed_dim, hidden_dim, hidden_dim * 2, TGT_PAD_IDX)

        def forward(self, src, tgt, teacher_forcing: float = 1.0):
            src_mask = src != SRC_PAD_IDX
            enc_outputs, hidden = self.encoder(src)
            prev_tok = tgt[:, 0]
            logits_by_t = []
            for t in range(1, tgt.size(1)):
                logits, hidden = self.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
                logits_by_t.append(logits.unsqueeze(1))
                predicted = logits.argmax(dim=1)
                prev_tok = tgt[:, t] if random.random() < teacher_forcing else predicted
            return torch.cat(logits_by_t, dim=1)

    def rebuild_vocabs_from_checkpoint(checkpoint: dict) -> None:
        global src_itos, src_stoi, tgt_itos, tgt_stoi, SRC_PAD_IDX, TGT_PAD_IDX
        src_itos = checkpoint["src_itos"]
        tgt_itos = checkpoint["tgt_itos"]
        src_stoi = {tok: i for i, tok in enumerate(src_itos)}
        tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}
        SRC_PAD_IDX = src_stoi[PAD]
        TGT_PAD_IDX = tgt_stoi[PAD]

    checkpoint_exists = CHECKPOINT_FILE.exists()
    model_embed_dim = EMBED_DIM
    model_hidden_dim = HIDDEN_DIM

    if not TRAIN_MODEL and checkpoint_exists:
        checkpoint = torch.load(CHECKPOINT_FILE, map_location=device)
        rebuild_vocabs_from_checkpoint(checkpoint)
        model_config = checkpoint.get("model_config", {})
        model_embed_dim = int(model_config.get("embed_dim", EMBED_DIM))
        model_hidden_dim = int(model_config.get("hidden_dim", HIDDEN_DIM))
        neural_model = Seq2Seq(len(src_itos), len(tgt_itos), model_embed_dim, model_hidden_dim).to(device)
        neural_model.load_state_dict(checkpoint["model_state"])
        NEURAL_READY = True
        checkpoint_saved_path = str(CHECKPOINT_FILE.relative_to(ROOT))
        print(f"Loaded checkpoint: {checkpoint_saved_path}")

    elif TRAIN_MODEL:
        torch.manual_seed(RANDOM_SEED)
        random.seed(RANDOM_SEED)
        neural_model = Seq2Seq(len(src_itos), len(tgt_itos), EMBED_DIM, HIDDEN_DIM).to(device)
        optimizer = torch.optim.Adam(neural_model.parameters(), lr=LEARNING_RATE)
        criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

        for epoch in range(1, EPOCHS + 1):
            neural_model.train()
            losses = []
            for src, tgt, _batch in make_batches(train_examples, BATCH_SIZE, shuffle=True):
                src = src.to(device)
                tgt = tgt.to(device)
                optimizer.zero_grad()
                logits = neural_model(src, tgt, teacher_forcing=TEACHER_FORCING)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(neural_model.parameters(), 1.0)
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            print(f"epoch {epoch:02d} train_loss={sum(losses) / max(1, len(losses)):.4f}")

        NEURAL_READY = True
        if SAVE_CHECKPOINT:
            CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
            torch.save(
                {
                    "model_state": neural_model.state_dict(),
                    "src_itos": src_itos,
                    "tgt_itos": tgt_itos,
                    "model_config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM},
                    "training_config": TRAINING_CONFIG,
                    "build_config": BUILD_CONFIG,
                    "factorization_config": FACTORIZATION_CONFIG.to_json(),
                },
                CHECKPOINT_FILE,
            )
            checkpoint_saved_path = str(CHECKPOINT_FILE.relative_to(ROOT))
            print(f"Saved checkpoint: {checkpoint_saved_path}")
    else:
        print("TRAIN_MODEL is False and no checkpoint exists; neural inference is unavailable.")

    if NEURAL_READY:
        @torch.no_grad()
        def neural_predict(text: str, max_len: int | None = None) -> list[str]:
            neural_model.eval()
            if max_len is None:
                max_len = min(MAX_TGT_LEN, MAX_DECODE_LEN)
            src_ids = encode_source(text)
            src = torch.tensor([src_ids], dtype=torch.long, device=device)
            src_mask = src != SRC_PAD_IDX
            enc_outputs, hidden = neural_model.encoder(src)
            prev_tok = torch.tensor([tgt_stoi[BOS]], dtype=torch.long, device=device)
            out_ids = []
            repeated_suffix_counts = Counter()
            for _ in range(max_len):
                logits, hidden = neural_model.decoder.step(prev_tok, hidden, enc_outputs, src_mask)
                next_id = int(logits.argmax(dim=1).item())
                if tgt_itos[next_id] == EOS:
                    break
                out_ids.append(next_id)
                if len(out_ids) >= 18:
                    suffix = tuple(out_ids[-6:])
                    repeated_suffix_counts[suffix] += 1
                    if repeated_suffix_counts[suffix] >= 3:
                        out_ids = out_ids[:-6]
                        break
                prev_tok = torch.tensor([next_id], dtype=torch.long, device=device)
            return decode_target(out_ids)

        print("Neural model is ready for evaluation/inference.")
elif TORCH_AVAILABLE:
    print("No selected rows are available after filtering; adjust length limits or rebuild data.")
else:
    def neural_predict(text: str, max_len: int | None = None) -> list[str]:
        raise RuntimeError("Torch is not installed; neural prediction is unavailable")

torch: 2.11.0
device: mps
epoch 01 train_loss=1.7384
epoch 02 train_loss=0.6605
epoch 03 train_loss=0.4903
Saved checkpoint: tokenizer/output/morph_tokenizer_poc.pt
Neural model is ready for evaluation/inference.


## 9. Evaluation and experiment history

Metrics are intentionally compact:

- `exact`: full target sequence match
- `token_acc`: position-wise token accuracy
- `token_f1`: multiset token F1 over the whole target
- `m_f1`: multiset F1 over `<M:...>` tokens only
- `g_f1`: multiset F1 over `<G:...>` tokens only
- `raw_rate`: share of baseline-like morpheme chunks emitted as `<RAW:...>`

Each evaluation appends one row to `tokenizer/output/morph_experiment_history.jsonl` when `WRITE_EXPERIMENT_HISTORY = True`.

In [10]:
eval_examples = dev_examples[:EVAL_LIMIT]
metrics = []

baseline_metrics = evaluate_prediction_fn("baseline", baseline_tokenize, eval_examples)
metrics.append(baseline_metrics)

if USE_NAVARRO_LEXICON:
    lexicon_metrics = evaluate_prediction_fn("lexicon_baseline", lexicon_baseline_tokenize, eval_examples)
    metrics.append(lexicon_metrics)

if NEURAL_READY:
    neural_metrics = evaluate_prediction_fn("neural", neural_predict, eval_examples)
    metrics.append(neural_metrics)
else:
    print("Neural evaluation skipped; train or load a checkpoint first.")

print(format_metrics_table(metrics))

history_row = {
    "timestamp": utc_now_iso(),
    "corpus_row_count": len(corpus_rows),
    "canonical_row_count": len(canonical_rows),
    "morph_row_count": len(morph_rows),
    "train_row_count": len(train_examples),
    "dev_row_count": len(dev_examples),
    "vocab_sizes": {
        "source_chars": len(src_itos),
        "target_tokens": len(tgt_itos),
        "m_tokens": len(morph_vocab["m_tokens"]),
        "g_tokens": len(morph_vocab["g_tokens"]),
    },
    "build_config": BUILD_CONFIG,
    "factorization_config": FACTORIZATION_CONFIG.to_json(),
    "lexicon_config": LEXICON_CONFIG,
    "navarro_lexicon_entry_count": len(navarro_entries),
    "navarro_augment_row_count": len(navarro_aug_rows),
    "training_config": TRAINING_CONFIG,
    "skip_counts": dict(skip_counts),
    "metrics": metrics,
    "checkpoint_path": checkpoint_saved_path,
}

if WRITE_EXPERIMENT_HISTORY:
    append_jsonl(HISTORY_FILE, history_row)
    print(f"Appended experiment history: {HISTORY_FILE.relative_to(ROOT)}")

model             exact  token_acc  token_f1  m_f1   g_f1   raw_rate
baseline          0.130  0.502      0.714     0.646  0.749  0.011   
lexicon_baseline  0.000  0.076      0.218     0.194  0.247  0.040   
neural            0.015  0.781      0.863     0.738  0.933  0.000   
Appended experiment history: tokenizer/output/morph_experiment_history.jsonl


In [11]:
print("Qualitative examples")
for row in eval_examples[:QUALITATIVE_EXAMPLES]:
    gold = row["target_tokens"]
    baseline_pred = baseline_tokenize(row["input"])
    lexicon_pred = lexicon_baseline_tokenize(row["input"])
    neural_pred = neural_predict(row["input"]) if NEURAL_READY else []
    print("\n" + "=" * 80)
    print("INPUT:   ", row["input"])
    print("GOLD:    ", compact_tokens(gold, max_tokens=100))
    print("BASELINE:", compact_tokens(baseline_pred, max_tokens=100))
    print("  mismatch:", mismatch_summary(baseline_pred, gold))
    print("LEXICON: ", compact_tokens(lexicon_pred, max_tokens=100))
    print("  mismatch:", mismatch_summary(lexicon_pred, gold))
    if NEURAL_READY:
        print("NEURAL:  ", compact_tokens(neural_pred, max_tokens=100))
        print("  mismatch:", mismatch_summary(neural_pred, gold))
    else:
        print("NEURAL:   unavailable")

Qualitative examples

INPUT:    a'e tereîpumĩ umẽ
GOLD:     <M:000120> <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000077> <G:2ps> <G:SUBJECT_PREFIX> <M:000051> <G:3p> <G:DEFAULT> <G:OBJECT_MARKER> <M:003776> <M:000062> <G:NEGATION_PARTICLE> <G:UME>
BASELINE: <M:000120> <M:000064> <G:ADVERSATIVE> <G:PARTICLE> <M:003933> <M:000051> <G:NEGATION_SUFFIX> <G:VOWEL_ENDING> <M:003776> <M:000062> <G:NEGATION_PARTICLE> <G:UME>
  mismatch: @1: pred=<M:000064> gold=<M:000017>; @2: pred=<G:ADVERSATIVE> gold=<G:PERMISSIVE_PREFIX>; @3: pred=<G:PARTICLE> gold=<G:VOWEL>; @4: pred=<M:003933> gold=<M:000077>; @5: pred=<M:000051> gold=<G:2ps>; @6: pred=<G:NEGATION_SUFFIX> gold=<G:SUBJECT_PREFIX>; @7: pred=<G:VOWEL_ENDING> gold=<M:000051>; @8: pred=<M:003776> gold=<G:3p>
LEXICON:  <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000445> <M:000017> <G:PERMISSIVE_PREFIX> <G:VOWEL> <M:000050> <M:000003> <G:PLURIFORM_PREFIX> <G:R> <M:000050> <M:000051> <G:NEGATION_SUFFIX> <G:VOWEL_ENDING> <M:003589> <M:005209> <M:0

## 10. Manual test strings

Edit `TEST_STRINGS` and rerun this cell. It shows baseline output, neural output when available, inspection groups, and RAW/OOV chunks.

In [12]:
TEST_STRINGS = [
    "amém",
    "tuba ta'yra Espírito Santo rera pupé",
    "orépysyrõte îepé mba'eaíba suí",
    "aîpotar nde kûara",
    "xe rera",
    "xerera",
]

for text in TEST_STRINGS:
    print("\n" + "=" * 80)
    print("INPUT:", text)

    baseline_pred = baseline_tokenize(text)
    baseline_raw_chunks = [tok for tok in baseline_pred if tok.startswith("<RAW:")]
    print("\nBASELINE:", compact_tokens(baseline_pred, max_tokens=140))
    print(f"baseline RAW/OOV rate: {baseline_raw_rate(baseline_pred):.2%}")
    print("RAW/OOV chunks:", baseline_raw_chunks or "none")
    print("inspection:")
    pprint(inspect_tokens(baseline_pred)[:40])

    lexicon_pred, lexicon_trace = lexicon_baseline_tokenize_with_trace(text)
    lexicon_raw_chunks = [tok for tok in lexicon_pred if tok.startswith("<RAW:")]
    print("\nLEXICON BASELINE:", compact_tokens(lexicon_pred, max_tokens=140))
    print(f"lexicon RAW/OOV rate: {baseline_raw_rate(lexicon_pred):.2%}")
    print("RAW/OOV chunks:", lexicon_raw_chunks or "none")
    print("inspection:")
    pprint(inspect_tokens(lexicon_pred)[:40])
    print("score trace:")
    pprint(lexicon_trace[:20])

    if NEURAL_READY:
        neural_pred = neural_predict(text)
        neural_raw_chunks = [tok for tok in neural_pred if tok.startswith("<RAW:")]
        print("\nNEURAL:", compact_tokens(neural_pred, max_tokens=140))
        print("RAW/OOV chunks:", neural_raw_chunks or "none")
        print("inspection:")
        pprint(inspect_tokens(neural_pred)[:40])
    else:
        print("\nNEURAL: unavailable; install torch and run the training/checkpoint cell.")


INPUT: amém

BASELINE: <M:000024> <G:AMEN> <G:INTERJECTION>
baseline RAW/OOV rate: 0.00%
RAW/OOV chunks: none
inspection:
[{'grammar': ['AMEN', 'INTERJECTION'],
  'raw': False,
  'surface': 'amém',
  'token': '<M:000024>'}]

LEXICON BASELINE: <M:000006> <G:1ps> <G:SUBJECT_PREFIX> <M:000094> <G:ABSOLUTE> <G:M> <G:PLURIFORM_PREFIX> <LEX:NAVARRO:noun:é> <G:NOUN> <G:ROOT> <M:000094> <G:ABSOLUTE> <G:M> <G:PLURIFORM_PREFIX>
lexicon RAW/OOV rate: 0.00%
RAW/OOV chunks: none
inspection:
[{'grammar': ['1ps', 'SUBJECT_PREFIX'],
  'raw': False,
  'surface': 'a',
  'token': '<M:000006>'},
 {'grammar': ['ABSOLUTE', 'M', 'PLURIFORM_PREFIX'],
  'raw': False,
  'surface': 'm',
  'token': '<M:000094>'},
 {'classname': 'noun',
  'definition': '(s.) - coisa distinta, coisa própria, coisa diferente; (adj.) '
                '- próprio, vário, outro, diferente, não comum aos outros, '
                'particular; separado (e não de parceria com alguém): Tub-é. - '
                'Tem outro pai (isto é, di

## Current limitations

This is still a proof of concept:

- the baseline is greedy longest-match, not a weighted finite-state segmenter
- spacing is only inspection-level reversible
- feature ordering is inherited from pydicate tag strings and may need a more explicit feature ontology
- the neural model is deliberately small and trained from scratch
- exact match is harsh because some forms have ambiguous analyses
- Navarro feature inference is conservative and based on obvious definition/gloss cues

The repeatable part is the important step: when you add pydicate-encoded Old Tupi data, set `FORCE_REBUILD = True`, rerun the notebook, and compare `morph_experiment_history.jsonl`. More rows should improve the factorized dataset, baseline coverage, target vocabulary, and eventually neural generalization.